In [ ]:
# 生成测试数据

#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
生成用于测试上传工具的模拟数据文件
运行后将创建 ./TestData 目录，包含：
  - 循环老化数据 (CSV)
  - 倍率性能数据 (CSV)
  - EIS阻抗数据 (CSV)
  - DSC热分析数据 (CSV)
  - 针刺温度数据 (CSV)
"""

import os
import csv
import math
import random

OUTPUT_DIR = "./TestData"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --------------------------------------------------------------
# 1. 常温循环数据：NMC811_25C_cycle.csv
# --------------------------------------------------------------
file1 = os.path.join(OUTPUT_DIR, "NMC811_25C_cycle.csv")
with open(file1, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Cycle", "Step", "Capacity(Ah)", "Energy(Wh)", "Voltage(V)", "Current(A)"])
    cap = 2.5
    for cycle in range(1, 201):   # 200个循环，模拟快速衰减
        cap -= 0.002 * random.uniform(0.8, 1.2)
        writer.writerow([cycle, "CC_Chg", round(cap*0.98, 4), round(cap*3.7, 4), 4.2, 1.25])
        writer.writerow([cycle, "CC_DChg", round(cap, 4), round(cap*3.2, 4), 2.5, -2.5])

# --------------------------------------------------------------
# 2. 倍率性能数据：Gr_25C_rate.csv
# --------------------------------------------------------------
file2 = os.path.join(OUTPUT_DIR, "Gr_25C_rate.csv")
with open(file2, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Rate(C)", "Discharge_Capacity(Ah)", "Average_Voltage(V)"])
    rates = [0.2, 0.5, 1, 2, 3]
    for r in rates:
        cap = round(2.5 * (1 - 0.05 * rates.index(r)), 3)
        volt = round(3.6 - 0.1 * r, 2)
        writer.writerow([r, cap, volt])

# --------------------------------------------------------------
# 3. EIS 数据：LCO_EIS.csv
# --------------------------------------------------------------
file3 = os.path.join(OUTPUT_DIR, "LCO_EIS.csv")
freqs = [100000, 10000, 1000, 100, 10, 1, 0.1, 0.01]
with open(file3, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Frequency(Hz)", "Z_real(Ohm)", "Z_imag(Ohm)"])
    for freq in freqs:
        real = 0.05 + 0.02 * math.log10(freq) + random.uniform(-0.001, 0.001)
        imag = -0.03 * math.log10(freq) - 0.01
        writer.writerow([freq, round(real, 5), round(imag, 5)])

# --------------------------------------------------------------
# 4. DSC 数据：NMC811_DSC.csv
# --------------------------------------------------------------
file4 = os.path.join(OUTPUT_DIR, "NMC811_DSC.csv")
with open(file4, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Temperature(℃)", "HeatFlow(mW/mg)"])
    for temp in range(30, 351, 2):
        hf = 0.05 + 0.1 * math.exp(-((temp-100)/20)**2) + random.uniform(-0.02, 0.02)
        writer.writerow([temp, round(hf, 4)])

# --------------------------------------------------------------
# 5. 针刺温度数据：nail_penetration_temp.csv
# --------------------------------------------------------------
file5 = os.path.join(OUTPUT_DIR, "nail_penetration_temp.csv")
with open(file5, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Time(s)", "T1_center(℃)", "T2_surface(℃)", "T3_tab(℃)"])
    for t in range(0, 1200, 1):
        if t < 300:
            t1, t2, t3 = 25.0, 25.0, 25.0
        else:
            t1 = 25 + 200 * (1 - math.exp(-(t-300)/50)) + random.uniform(-2, 2)
            t2 = 25 + 180 * (1 - math.exp(-(t-300)/60)) + random.uniform(-1, 1)
            t3 = 25 + 150 * (1 - math.exp(-(t-300)/70))
        writer.writerow([t, round(t1, 1), round(t2, 1), round(t3, 1)])

print(f"模拟数据文件已生成至：{os.path.abspath(OUTPUT_DIR)}")
print("请使用 data_uploader.py 依次上传这些文件进行测试。")

模拟数据文件已生成至：e:\Users\03清华PSG\05昱姐课题\02Pro\20260717_表格搜集\TestData
请使用 data_uploader.py 依次上传这些文件进行测试。


In [3]:
!python data_uploader.py

libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile


In [ ]:
!python experiment_locator.py

libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile


In [2]:
#!/usr/bin/env python3
# 生成循环性能模拟数据（放电容量严格单调下降）
import os
import csv
import random

OUTPUT_DIR = "./TestData_Cycle"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 样品列表： (样品编号, 初始放电比容量（第2圈）, 最终容量, 循环数)
samples = [
    ("NMC811-C01", 116.0, 49.0, 99),   # 从116衰减到49
    ("NMC811-C02", 118.0, 56.0, 99),
    ("NMC532-C03", 114.0, 69.0, 99)    # 衰减较慢
]

for name, init_cap, final_cap, n_cycles in samples:
    filename = f"{name}_cycle.csv"
    filepath = os.path.join(OUTPUT_DIR, filename)
    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["循环号", "放电比容量(mAh/g)", "充电比容量(mAh/g)", "充放电效率"])
        prev_cap = init_cap  # 上一圈的放电容量，初始为第2圈的起始值
        for cycle in range(2, n_cycles + 1):
            # 计算理想线性衰减位置
            progress = (cycle - 2) / (n_cycles - 1)  # 0 到 1
            target_cap = init_cap - (init_cap - final_cap) * progress
            # 添加随机波动，但波动范围只允许在 [-0.5, 0] 之间，确保容量不回升
            noise = random.uniform(-0.6, 0.0)
            cap = target_cap + noise
            # 严格保证容量 <= 前一圈，且 > 0
            cap = max(0.1, min(cap, prev_cap - 0.001))
            prev_cap = cap  # 更新为当前圈容量，作为下一圈的上一圈
            # 生成充电容量和效率
            eff = random.uniform(98.0, 102.0)
            if cycle <= 5:
                eff = random.uniform(95.0, 101.5)
            charge_cap = cap * (100.0 / eff) if eff != 0 else cap
            efficiency = (cap / charge_cap) * 100 if charge_cap != 0 else 100.0
            writer.writerow([cycle, round(cap, 4), round(charge_cap, 4), round(efficiency, 4)])
    print(f"生成 {filename}")

print(f"循环数据已保存至 {os.path.abspath(OUTPUT_DIR)}")

生成 NMC811-C01_cycle.csv
生成 NMC811-C02_cycle.csv
生成 NMC532-C03_cycle.csv
循环数据已保存至 e:\Users\03清华PSG\05昱姐课题\02Pro\20260717_表格搜集\TestData_Cycle


In [2]:
#!/usr/bin/env python3
# 生成 EIS 模拟数据，用于测试提交和对比

import os
import csv
import numpy as np

OUTPUT_DIR = "./TestData_EIS"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 频率范围 (Hz)
freqs = np.logspace(5, -2, 60)  # 100kHz 到 0.01Hz

# 样品列表： (样品名, Rs, Rct, CPE_T, CPE_P, W_R, W_T, W_P)
# 简单等效电路： Rs + (Rct//CPE) + Warburg_open
samples = [
    ("LCO-M01", 2.0, 5.0, 1e-4, 0.8, 10.0, 0.5, 0.5),
    ("LCO-M02", 2.5, 7.0, 1.2e-4, 0.8, 12.0, 0.6, 0.5),
    ("NMC811-M03", 1.8, 4.0, 9e-5, 0.85, 8.0, 0.4, 0.5)
]

def compute_impedance(freq, Rs, Rct, CPE_T, CPE_P, W_R, W_T, W_P):
    omega = 2 * np.pi * freq
    # CPE: Zcpe = 1 / (T*(jω)^P)
    Z_CPE = 1.0 / (CPE_T * (1j * omega) ** CPE_P)
    # Rct 并联 CPE
    Z_parallel = (Rct * Z_CPE) / (Rct + Z_CPE)
    # Warburg (open): Z_w = W_R * coth((jω*W_T)^W_P) / (jω*W_T)^W_P  简化用扩散阻抗
    # 使用 Z_w = W_R * (jω*W_T)^(-0.5) 的常见形式
    Z_W = W_R * (1j * omega * W_T) ** (-W_P)
    Z_total = Rs + Z_parallel + Z_W
    return Z_total

for name, Rs, Rct, CPE_T, CPE_P, W_R, W_T, W_P in samples:
    filename = f"{name}_EIS.csv"
    filepath = os.path.join(OUTPUT_DIR, filename)
    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Freq/Hz", "Z'/ohm", "Z\"/ohm", "Z/ohm", "Phase/deg"])
        for freq in freqs:
            Z = compute_impedance(freq, Rs, Rct, CPE_T, CPE_P, W_R, W_T, W_P)
            real, imag = np.real(Z), np.imag(Z)
            magnitude = np.abs(Z)
            phase = np.angle(Z, deg=True)
            writer.writerow([f"{freq:.4e}", f"{real:.6e}", f"{imag:.6e}", f"{magnitude:.6e}", f"{phase:.4f}"])
    print(f"生成 {filename}")

print(f"EIS 数据已保存至 {os.path.abspath(OUTPUT_DIR)}")

生成 LCO-M01_EIS.csv
生成 LCO-M02_EIS.csv
生成 NMC811-M03_EIS.csv
EIS 数据已保存至 e:\Users\03清华PSG\05昱姐课题\02Pro\20260717_表格搜集\TestData_EIS


In [3]:
# 测试
!python data_uploader.py

In [ ]:
!python experiment_locator.py